# 角度直接测量与不确定度评定

本 Notebook 用于对分光计等仪器测得的角度多次测量数据（支持度分格式 `度.分`）进行不确定度合成与保留一位有效数字的规范修约输出。

---
### 实验原理与数学公式

#### 1. 角度单位换算与平均值计算
角度采用“分（arcminute）”为内部基准单位换算：
$$D^\circ M' = D \times 60' + M'$$
对 $n$ 次测量值 $\theta_1, \theta_2, \dots, \theta_n$（以分为单位）：
$$\bar{\theta} = \frac{1}{n} \sum_{i=1}^n \theta_i$$
$$s = \sqrt{\frac{1}{n-1} \sum_{i=1}^n (\theta_i - \bar{\theta})^2}$$

#### 2. 不确定度评定公式
* **A 类不确定度**：
$$u_A = \frac{s}{\sqrt{n}}$$
* **B 类不确定度**（分光计游标盘分度值误差 $\Delta_{\mathrm{inst}}$ 通常为 $1'$）：
$$u_B = \frac{\Delta_{\mathrm{inst}}}{\sqrt{3}}$$
* **合成不确定度**：
$$u = \sqrt{u_A^2 + u_B^2}$$

#### 3. 结果修约规则
不确定度 $u$ 按照“四舍六入五凑偶”保留一位有效数字，角度平均值末位与不确定度对齐，并还原输出为度分格式 $D^\circ M'$。

In [ ]:
import math
from decimal import Decimal
from python.utils import parse_angle_to_minutes, format_minutes_as_angle, scientific_round

print("角度换算与修约模块加载完成。")

### 1. 角度测量数据与分度值输入
> **输入格式**：字符串 `'度.分'`（例如 `'120.15'` 代表 $120^\circ 15'$）。

In [ ]:
# 测量数据列表 (度.分 格式)
angle_data = ["120.15", "120.16", "120.14", "120.15", "120.17", "120.15"]

# 仪器分度值 (单位: 分 ')
res_str = "1"

print(f"输入角度数据: {angle_data}")
print(f"仪器分度值: {res_str} '")

### 2. 统计计算与修约输出

In [ ]:
# 转换为内部单位“分”
data_minutes = [parse_angle_to_minutes(s) for s in angle_data]
res = Decimal(res_str)
n = len(data_minutes)

# 1. 平均值与标准差
mean_minutes = sum(data_minutes) / n
variance = sum((x - mean_minutes) ** 2 for x in data_minutes) / (n - 1) if n > 1 else Decimal("0")
s = Decimal(str(math.sqrt(float(variance))))

# 2. A类、B类及合成不确定度
u_A = s / Decimal(str(math.sqrt(n))) if n > 1 else Decimal("0")
u_B = res / Decimal(str(math.sqrt(3)))
u = Decimal(str(math.sqrt(float(u_A**2 + u_B**2))))

# 3. 科学修约与格式还原
u_float = float(u)
if u_float == 0:
    final_result_str = f"θ = {format_minutes_as_angle(mean_minutes)} ± 0'"
else:
    first_digit_pos = math.floor(math.log10(u_float))
    prec = Decimal('1e' + str(first_digit_pos))
    mean_final_val, u_final = scientific_round(mean_minutes, u)
    mean_str = format_minutes_as_angle(mean_final_val, prec)
    final_result_str = f"θ = {mean_str} ± {u_final}'"

print("=" * 45)
print("         角 度 测 量 不 确 定 度 分 析         ")
print("=" * 45)
print(f"测量次数 n         : {n}")
print(f"角度平均值         : {format_minutes_as_angle(mean_minutes)} ({mean_minutes:.4f}')")
print(f"样本标准差 s       : {s:.6f}'")
print(f"A 类不确定度 u_A   : {u_A:.6f}'")
print(f"B 类不确定度 u_B   : {u_B:.6f}'")
print(f"合成不确定度 u     : {u:.6f}'")
print("-" * 45)
print(f"★ 最终修约结果    : {final_result_str}")
print("=" * 45)